# 1.3 知识蒸馏（Knowledge Distillation）

## 什么是知识蒸馏？

知识蒸馏是将大模型（教师模型）的知识迁移到小模型（学生模型）的技术。学生模型通过模仿教师模型的输出分布、中间特征或注意力模式，在参数量大幅减少的情况下仍能逼近大模型的性能。

## 为什么用知识蒸馏？

1. **压缩比大**：学生模型可以是教师模型的1/10甚至更小，同时保持大部分性能。例如DistilBERT将BERT压缩到66%大小，保留97%的性能。
2. **暗知识传递**：教师模型的soft output包含类别间的相似性信息（"暗知识"），比硬标签信息更丰富，帮助学生学到更好的表示。
3. **与量化/剪枝互补**：蒸馏可以与量化和剪枝叠加使用，先蒸馏得到小模型，再量化/剪枝进一步压缩。
4. **端侧部署关键**：蒸馏后的小模型适合端侧部署，推理速度快、内存占用低。

## 知识蒸馏的分类

根据是否可访问教师模型内部状态：
- **白盒蒸馏**：可访问教师模型的logits、中间层特征、注意力图等内部状态
- **黑盒蒸馏**：仅能通过API获取教师模型的输出（如生成的文本）

### 知识蒸馏的核心数学框架

知识蒸馏的通用损失函数：
$$L = \alpha \cdot L_{KD}(f_s, f_t) + (1 - \alpha) \cdot L_{CE}(y_s, y_{true})$$

其中：
- $f_s$：学生模型的输出
- $f_t$：教师模型的输出
- $L_{KD}$：蒸馏损失，衡量学生与教师的差异
- $L_{CE}$：标准交叉熵损失，确保学生模型正确分类
- $\alpha$：蒸馏损失权重，平衡蒸馏目标和分类目标

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

---
## 1.3.1 白盒蒸馏（White-Box Distillation）

### 什么是白盒蒸馏？

白盒蒸馏可以访问教师模型的内部状态（logits、中间层特征、注意力图等），利用这些丰富信息指导学生模型训练。由于信息更丰富，白盒蒸馏通常比黑盒蒸馏效果更好。

### 为什么白盒蒸馏更有效？

1. **丰富的监督信号**：教师模型的中间层特征和注意力图提供了比最终输出更细粒度的监督信息。
2. **暗知识传递**：教师的soft logits包含类别间的相似性信息，例如"猫和狗比猫和汽车更相似"，这些信息硬标签无法提供。
3. **多层次对齐**：可以在多个层级（logits、特征、注意力）同时蒸馏，提供更全面的指导。

### Logits级蒸馏（Response-Based）

#### 什么是Logits级蒸馏？

学生模型的输出logits与教师模型的soft logits之间计算KL散度损失。教师输出的概率分布包含"暗知识"（类别间的相似性信息），比硬标签信息更丰富。这是Hinton等人2015年提出的经典方法。

#### 为什么温度参数重要？

温度$T$控制softmax的平滑程度。$T=1$时为标准softmax，$T$越大概率分布越平滑，暗知识越明显。例如，$T=10$时，原本概率为0.001的类别可能变为0.05，学生可以看到更多类别间的关系。

#### Logits级蒸馏的数学公式

$$L_{KD} = \alpha \cdot T^2 \cdot KL(\sigma(z_s/T) || \sigma(z_t/T)) + (1-\alpha) \cdot L_{CE}(y_s, y_{true})$$

**Softmax with Temperature**：
$$\sigma(z_i/T) = \frac{\exp(z_i/T)}{\sum_j \exp(z_j/T)}$$

其中：
- $z_s$：学生模型的logits（未归一化输出）
- $z_t$：教师模型的logits
- $T$：温度参数，$T > 1$使分布更平滑，暗知识更丰富
- $\sigma(z/T)$：温度缩放后的softmax概率分布
- $KL(\cdot || \cdot)$：KL散度，衡量两个分布的差异
- $T^2$：补偿因子，因为温度缩放导致梯度缩小$T^2$倍，需要乘回来
- $\alpha$：蒸馏损失权重，通常取0.5-0.9
- $L_{CE}$：标准交叉熵损失
- $y_{true}$：真实标签

In [ ]:
class TeacherModel(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        return self.net(x)

class StudentModel(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 32), nn.ReLU(),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        return self.net(x)

def distillation_loss(student_logits, teacher_logits, labels, temperature=4.0, alpha=0.7):
    """知识蒸馏损失 = α·T²·KL散度 + (1-α)·交叉熵
    F.kl_div(input, target): input=log概率(student), target=概率(teacher)"""
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    soft_teacher = F.softmax(teacher_logits / temperature, dim=-1)
    kl_loss = F.kl_div(soft_student, soft_teacher.detach(), reduction='batchmean') * (temperature ** 2)
    ce_loss = F.cross_entropy(student_logits, labels)
    return alpha * kl_loss + (1 - alpha) * ce_loss

def train_with_distillation(teacher, student, train_loader, epochs=50,
                            temperature=4.0, alpha=0.7, lr=1e-3):
    """使用知识蒸馏训练学生模型"""
    teacher.eval()
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)
    loss_history = []

    for epoch in range(epochs):
        epoch_loss = 0
        for x, y in train_loader:
            with torch.no_grad():
                teacher_logits = teacher(x)
            student_logits = student(x)
            loss = distillation_loss(student_logits, teacher_logits, y,
                                     temperature=temperature, alpha=alpha)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(train_loader))
    return loss_history

def train_normal(model, train_loader, epochs=50, lr=1e-3):
    """普通训练（无蒸馏）"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_history = []
    for epoch in range(epochs):
        epoch_loss = 0
        for x, y in train_loader:
            output = model(x)
            loss = F.cross_entropy(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(train_loader))
    return loss_history

dim, n_classes = 64, 10
x_data = torch.randn(512, dim)
y_data = torch.randint(0, n_classes, (512,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=64)

teacher = TeacherModel(dim, n_classes)
train_normal(teacher, loader, epochs=30)

student_kd = StudentModel(dim, n_classes)
student_no_kd = StudentModel(dim, n_classes)

loss_kd = train_with_distillation(teacher, student_kd, loader, epochs=30, temperature=4.0)
loss_no_kd = train_normal(student_no_kd, loader, epochs=30)

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student_kd.parameters())

with torch.no_grad():
    acc_teacher = (teacher(x_data).argmax(1) == y_data).float().mean()
    acc_kd = (student_kd(x_data).argmax(1) == y_data).float().mean()
    acc_no_kd = (student_no_kd(x_data).argmax(1) == y_data).float().mean()

print(f"=== Logits级蒸馏效果 ===")
print(f"教师模型参数量: {teacher_params:,}, 准确率: {acc_teacher:.4f}")
print(f"学生模型参数量: {student_params:,} ({student_params/teacher_params:.1%} of teacher)")
print(f"学生(有蒸馏)准确率: {acc_kd:.4f}")
print(f"学生(无蒸馏)准确率: {acc_no_kd:.4f}")
print(f"蒸馏提升: {(acc_kd - acc_no_kd)*100:.2f}%")

print(f"\n=== 温度参数影响 ===")
for T in [1.0, 2.0, 4.0, 8.0, 16.0]:
    s = StudentModel(dim, n_classes)
    train_with_distillation(teacher, s, loader, epochs=30, temperature=T)
    with torch.no_grad():
        acc = (s(x_data).argmax(1) == y_data).float().mean()
    print(f"T={T:<5} 准确率: {acc:.4f}")

### 特征级蒸馏（Feature-Level / Intermediate-Layer）

#### 什么是特征级蒸馏？

特征级蒸馏对齐教师和学生中间层的隐藏状态或特征表示，让学生模型不仅学习教师的最终输出，还学习教师的中间表示。通常需要线性变换层将学生的特征映射到教师的特征空间。

#### 为什么特征级蒸馏更有效？

1. **更丰富的梯度信号**：中间层特征提供了比logits更细粒度的监督，梯度信号更丰富，训练更稳定。
2. **表示空间对齐**：强制学生的中间表示接近教师的表示，使学生学到更有区分力的特征。
3. **适合深层模型**：对于很深的网络，仅用logits蒸馏可能导致梯度消失，特征级蒸馏可以缓解这个问题。

#### 特征级蒸馏的数学公式

$$L_{feat} = \sum_{l} \|\phi_s^l(f_s^l) - f_t^l\|_2^2$$

总损失：
$$L = \alpha \cdot L_{KD} + \beta \cdot L_{feat} + (1-\alpha-\beta) \cdot L_{CE}$$

其中：
- $f_s^l$：学生模型第$l$层的特征
- $f_t^l$：教师模型第$l$层的特征
- $\phi_s^l$：对齐层，将学生特征映射到教师特征空间（通常是线性变换）
- $L_{feat}$：特征蒸馏损失，通常用MSE
- $\beta$：特征蒸馏损失权重
- 对齐层$\phi_s^l$是可训练的，维度为$\mathbb{R}^{d_s \times d_t}$，$d_s < d_t$

In [ ]:
class TeacherWithFeatures(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Linear(dim, 256), nn.ReLU())
        self.layer2 = nn.Sequential(nn.Linear(256, 128), nn.ReLU())
        self.layer3 = nn.Sequential(nn.Linear(128, 64), nn.ReLU())
        self.head = nn.Linear(64, n_classes)

    def forward(self, x, return_features=False):
        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        out = self.head(f3)
        if return_features:
            return out, [f1, f2, f3]
        return out

class StudentWithFeatures(nn.Module):
    def __init__(self, dim=64, n_classes=10):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Linear(dim, 64), nn.ReLU())
        self.layer2 = nn.Sequential(nn.Linear(64, 32), nn.ReLU())
        self.layer3 = nn.Sequential(nn.Linear(32, 16), nn.ReLU())
        self.head = nn.Linear(16, n_classes)
        self.align1 = nn.Linear(64, 256)
        self.align2 = nn.Linear(32, 128)
        self.align3 = nn.Linear(16, 64)

    def forward(self, x, return_features=False):
        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        out = self.head(f3)
        if return_features:
            aligned_f1 = self.align1(f1)
            aligned_f2 = self.align2(f2)
            aligned_f3 = self.align3(f3)
            return out, [aligned_f1, aligned_f2, aligned_f3]
        return out

def feature_distillation_loss(student_out, teacher_out, student_features, teacher_features,
                              labels, temperature=4.0, alpha=0.5, beta=0.3):
    """特征级蒸馏损失 = α·logits蒸馏 + β·特征蒸馏 + (1-α-β)·CE"""
    soft_s = F.log_softmax(student_out / temperature, dim=-1)
    soft_t = F.softmax(teacher_out / temperature, dim=-1)
    kl_loss = F.kl_div(soft_s, soft_t.detach(), reduction='batchmean') * (temperature ** 2)
    ce_loss = F.cross_entropy(student_out, labels)

    feat_loss = 0
    for sf, tf in zip(student_features, teacher_features):
        feat_loss += F.mse_loss(sf, tf.detach())
    feat_loss /= len(student_features)

    return alpha * kl_loss + beta * feat_loss + (1 - alpha - beta) * ce_loss

def train_feature_distillation(teacher, student, train_loader, epochs=30,
                               temperature=4.0, alpha=0.5, beta=0.3, lr=1e-3):
    """特征级蒸馏训练"""
    teacher.eval()
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)
    loss_history = []

    for epoch in range(epochs):
        epoch_loss = 0
        for x, y in train_loader:
            with torch.no_grad():
                t_out, t_feats = teacher(x, return_features=True)
            s_out, s_feats = student(x, return_features=True)
            loss = feature_distillation_loss(s_out, t_out, s_feats, t_feats, y,
                                             temperature=temperature, alpha=alpha, beta=beta)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(train_loader))
    return loss_history

teacher_feat = TeacherWithFeatures(dim, n_classes)
train_normal(teacher_feat, loader, epochs=30)

student_feat = StudentWithFeatures(dim, n_classes)
train_feature_distillation(teacher_feat, student_feat, loader, epochs=30)

student_logits_only = StudentModel(dim, n_classes)
train_with_distillation(teacher, student_logits_only, loader, epochs=30)

with torch.no_grad():
    acc_feat = (student_feat(x_data).argmax(1) == y_data).float().mean()
    acc_logits = (student_logits_only(x_data).argmax(1) == y_data).float().mean()

print(f"=== 特征级蒸馏 vs Logits级蒸馏 ===")
print(f"Logits级蒸馏准确率: {acc_logits:.4f}")
print(f"特征级蒸馏准确率: {acc_feat:.4f}")
print(f"\n特征级蒸馏优势: 利用中间层信息，提供更丰富的梯度信号")
print(f"代价: 需要额外的对齐层，训练更复杂")

### 注意力蒸馏（Attention Transfer）

#### 什么是注意力蒸馏？

让学生模型的注意力图模仿教师模型的注意力分布，学习教师模型的注意力模式。注意力图反映了模型"看哪里"的决策过程，是一种高度浓缩的知识。

#### 为什么注意力蒸馏有效？

1. **决策过程迁移**：注意力图揭示了模型的决策依据，学生通过模仿注意力模式可以学到更好的特征选择策略。
2. **空间信息保留**：注意力图包含空间关系信息，对视觉任务和序列建模任务特别有效。
3. **参数效率**：注意力图通常维度较小，蒸馏开销低。

#### 注意力蒸馏的数学公式

**注意力图提取**：
$$A^h = \text{softmax}\left(\frac{Q^h (K^h)^T}{\sqrt{d_k}}\right)$$

**注意力蒸馏损失**：
$$L_{attn} = \sum_{h} \|A_s^h - A_t^h\|_F^2$$

其中：
- $A_s^h, A_t^h$：学生和教师第$h$个头的注意力概率矩阵
- $Q^h, K^h$：第$h$个头的查询和键向量
- $d_k$：每个头的维度
- $\|\cdot\|_F^2$：Frobenius范数的平方
- 当学生和教师头数不同时，需要对教师的注意力图进行聚合或池化

In [ ]:
class AttentionModel(nn.Module):
    """支持注意力图提取的模型（使用hook）"""
    def __init__(self, dim=64, n_heads=4, n_classes=10):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.head = nn.Linear(dim, n_classes)
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self._attn_weights = None

    def _extract_attn(self, x):
        """手动提取多头注意力图: (B, n_heads, N, N)"""
        B, N, C = x.shape
        qkv = F.linear(x, self.attn.in_proj_weight, self.attn.in_proj_bias)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn_probs = F.softmax(attn_weights, dim=-1)
        return attn_probs

    def forward(self, x, return_attn=False):
        attn_probs = self._extract_attn(x)
        v_input = x
        B, N, C = x.shape
        qkv = F.linear(x, self.attn.in_proj_weight, self.attn.in_proj_bias)
        _, _, v = qkv.chunk(3, dim=-1)
        v = v.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        out = torch.matmul(attn_probs, v)
        out = out.transpose(1, 2).contiguous().view(B, N, C)
        out = self.attn.out_proj(out)
        logits = self.head(out.mean(dim=1))
        if return_attn:
            return logits, attn_probs
        return logits

def attention_distillation_loss(s_attn, t_attn, student_logits, labels, gamma=0.1):
    """注意力蒸馏损失"""
    s_attn_sum = s_attn.sum(dim=1, keepdim=True)
    t_attn_sum = t_attn.sum(dim=1, keepdim=True)
    n_s, n_t = s_attn.size(1), t_attn.size(1)
    if n_s != n_t:
        t_attn_pooled = t_attn_sum.expand(-1, n_s, -1, -1) / n_s
        s_attn_pooled = s_attn
    else:
        t_attn_pooled = t_attn
        s_attn_pooled = s_attn
    min_len = min(s_attn_pooled.size(-1), t_attn_pooled.size(-1))
    s_attn_crop = s_attn_pooled[:, :, :min_len, :min_len]
    t_attn_crop = t_attn_pooled[:, :, :min_len, :min_len]
    attn_loss = F.mse_loss(s_attn_crop, t_attn_crop.detach())
    ce_loss = F.cross_entropy(student_logits, labels)
    return ce_loss + gamma * attn_loss

x_seq = torch.randn(64, 8, 64)
y_seq = torch.randint(0, 10, (64,))
seq_dataset = torch.utils.data.TensorDataset(x_seq, y_seq)
seq_loader = torch.utils.data.DataLoader(seq_dataset, batch_size=16)

attn_teacher = AttentionModel(dim=64, n_heads=4, n_classes=10)
attn_student = AttentionModel(dim=64, n_heads=2, n_classes=10)

optimizer_t = torch.optim.Adam(attn_teacher.parameters(), lr=1e-3)
for _ in range(20):
    for x, y in seq_loader:
        loss = F.cross_entropy(attn_teacher(x), y)
        optimizer_t.zero_grad()
        loss.backward()
        optimizer_t.step()

attn_teacher.eval()
optimizer_s = torch.optim.Adam(attn_student.parameters(), lr=1e-3)
for _ in range(20):
    for x, y in seq_loader:
        with torch.no_grad():
            _, t_attn = attn_teacher(x, return_attn=True)
        s_logits, s_attn = attn_student(x, return_attn=True)
        loss = attention_distillation_loss(s_attn, t_attn, s_logits, y, gamma=0.1)
        optimizer_s.zero_grad()
        loss.backward()
        optimizer_s.step()

with torch.no_grad():
    acc_attn = (attn_student(x_seq).argmax(1) == y_seq).float().mean()
    acc_teacher = (attn_teacher(x_seq).argmax(1) == y_seq).float().mean()

print(f"=== 注意力蒸馏效果 ===")
print(f"教师(4头)准确率: {acc_teacher:.4f}")
print(f"学生(2头+注意力蒸馏)准确率: {acc_attn:.4f}")
print(f"\n注意力蒸馏让学生学会教师的注意力模式，提升小模型性能")

---
## 1.3.2 黑盒蒸馏（Black-Box Distillation）

### 什么是黑盒蒸馏？

黑盒蒸馏仅能通过API访问教师模型的输出（生成的文本），无法获取内部状态（logits、特征等）。通过教师模型生成的高质量数据来训练学生模型。

### 为什么用黑盒蒸馏？

1. **无需模型访问**：很多商业LLM（如GPT-4）只提供API接口，无法获取内部状态，黑盒蒸馏是唯一选择。
2. **数据增强**：教师模型可以生成大量高质量合成数据，比人工标注更高效。
3. **隐私保护**：不需要部署教师模型，避免泄露模型参数。

### 黑盒蒸馏的局限性

- 无法利用soft logits中的暗知识，只能使用hard labels
- 合成数据的质量和多样性直接影响蒸馏效果
- API调用成本可能很高

### 指令蒸馏（Instruction Distillation）

#### 什么是指令蒸馏？

使用教师模型对大量指令生成回答，构建指令微调数据集训练学生模型。如Alpaca、Vicuna等方法。

#### 指令蒸馏的流程

1. **指令收集**：收集大量多样化的指令（自指令生成或人工编写）
2. **教师回答**：用大模型（如GPT-4）对每条指令生成高质量回答
3. **数据过滤**：过滤低质量回答，保留高质量数据
4. **学生训练**：用指令-回答对微调学生模型

#### 数学公式

$$L = -\sum_{t} \log P_s(y_t | x_{<t}, \text{instruction})$$

其中：
- $P_s$：学生模型的条件概率
- $y_t$：教师生成的第$t$个token
- $x_{<t}$：第$t$个token之前的所有token
- instruction：输入指令

In [ ]:
class SimpleLLM(nn.Module):
    """简化版语言模型，用于演示指令蒸馏"""
    def __init__(self, vocab_size=1000, dim=64, n_classes=10):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=dim, nhead=4, batch_first=True),
            num_layers=2
        )
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        h = self.emb(x)
        h = self.transformer(h)
        return self.head(h.mean(dim=1))

vocab_size = 1000
teacher_llm = SimpleLLM(vocab_size, dim=64, n_classes=10)
student_llm = SimpleLLM(vocab_size, dim=32, n_classes=10)

optimizer_t = torch.optim.Adam(teacher_llm.parameters(), lr=1e-3)
x_tokens = torch.randint(0, vocab_size, (256, 16))
y_labels = torch.randint(0, 10, (256,))
token_dataset = torch.utils.data.TensorDataset(x_tokens, y_labels)
token_loader = torch.utils.data.DataLoader(token_dataset, batch_size=32)

for _ in range(20):
    for x, y in token_loader:
        loss = F.cross_entropy(teacher_llm(x), y)
        optimizer_t.zero_grad()
        loss.backward()
        optimizer_t.step()

print(f"=== 黑盒指令蒸馏 ===")
print(f"步骤1: 使用教师模型生成标注数据")

n_synthetic = 512
x_synthetic = torch.randint(0, vocab_size, (n_synthetic, 16))
with torch.no_grad():
    y_synthetic = teacher_llm(x_synthetic).argmax(dim=1)

print(f"生成合成数据: {n_synthetic}条")
print(f"合成标签分布: {torch.bincount(y_synthetic, minlength=10)}")

syn_dataset = torch.utils.data.TensorDataset(x_synthetic, y_synthetic)
syn_loader = torch.utils.data.DataLoader(syn_dataset, batch_size=32)

print(f"\n步骤2: 使用合成数据训练学生模型")
optimizer_s = torch.optim.Adam(student_llm.parameters(), lr=1e-3)
for _ in range(20):
    for x, y in syn_loader:
        loss = F.cross_entropy(student_llm(x), y)
        optimizer_s.zero_grad()
        loss.backward()
        optimizer_s.step()

student_no_distill = SimpleLLM(vocab_size, dim=32, n_classes=10)
optimizer_n = torch.optim.Adam(student_no_distill.parameters(), lr=1e-3)
for _ in range(20):
    for x, y in token_loader:
        loss = F.cross_entropy(student_no_distill(x), y)
        optimizer_n.zero_grad()
        loss.backward()
        optimizer_n.step()

with torch.no_grad():
    acc_teacher = (teacher_llm(x_tokens).argmax(1) == y_labels).float().mean()
    acc_student_kd = (student_llm(x_tokens).argmax(1) == y_labels).float().mean()
    acc_student_no = (student_no_distill(x_tokens).argmax(1) == y_labels).float().mean()

teacher_p = sum(p.numel() for p in teacher_llm.parameters())
student_p = sum(p.numel() for p in student_llm.parameters())

print(f"\n=== 黑盒蒸馏结果 ===")
print(f"教师模型: {teacher_p:,}参数, 准确率={acc_teacher:.4f}")
print(f"学生(黑盒蒸馏): {student_p:,}参数, 准确率={acc_student_kd:.4f}")
print(f"学生(无蒸馏): {student_p:,}参数, 准确率={acc_student_no:.4f}")
print(f"\n黑盒蒸馏关键: 合成数据的质量和多样性决定蒸馏效果")

### 自蒸馏（Self-Distillation）

#### 什么是自蒸馏？

模型自身作为教师，通过不同训练阶段或不同数据增强下的输出一致性来提升性能。常见做法是使用EMA（指数移动平均）模型作为教师，当前模型作为学生。

#### 为什么自蒸馏有效？

1. **无需外部教师**：不需要更大的教师模型，训练成本更低。
2. **EMA更稳定**：EMA模型是历史参数的平滑平均，输出更稳定、更准确，作为教师提供更好的监督信号。
3. **正则化效果**：自蒸馏相当于一种正则化，防止模型过拟合。

#### 自蒸馏的数学公式

**EMA更新**：
$$\theta_{ema} \leftarrow \mu \cdot \theta_{ema} + (1 - \mu) \cdot \theta_{model}$$

**自蒸馏损失**：
$$L = \alpha \cdot T^2 \cdot KL(\sigma(z/T; \theta) || \sigma(z/T; \theta_{ema})) + (1-\alpha) \cdot L_{CE}$$

其中：
- $\theta$：当前模型参数（学生）
- $\theta_{ema}$：EMA模型参数（教师）
- $\mu$：EMA衰减率，通常取0.99或0.999
- $z$：模型logits
- $T$：温度参数

In [ ]:
class SelfDistillationTrainer:
    """自蒸馏训练器：使用EMA模型作为教师"""
    def __init__(self, model, ema_decay=0.99, temperature=4.0, alpha=0.5):
        self.model = model
        self.ema_decay = ema_decay
        self.temperature = temperature
        self.alpha = alpha
        self.ema_model = copy.deepcopy(model)
        for p in self.ema_model.parameters():
            p.requires_grad = False

    def update_ema(self):
        with torch.no_grad():
            for p_ema, p_model in zip(self.ema_model.parameters(), self.model.parameters()):
                p_ema.data = self.ema_decay * p_ema.data + (1 - self.ema_decay) * p_model.data

    def train_step(self, x, y):
        student_logits = self.model(x)
        with torch.no_grad():
            teacher_logits = self.ema_model(x)

        soft_s = F.log_softmax(student_logits / self.temperature, dim=-1)
        soft_t = F.softmax(teacher_logits / self.temperature, dim=-1)
        kl_loss = F.kl_div(soft_s, soft_t, reduction='batchmean') * (self.temperature ** 2)
        ce_loss = F.cross_entropy(student_logits, y)
        loss = self.alpha * kl_loss + (1 - self.alpha) * ce_loss
        return loss

model_self = StudentModel(dim=64, n_classes=10)
trainer = SelfDistillationTrainer(model_self, ema_decay=0.99, temperature=4.0, alpha=0.5)
optimizer = torch.optim.Adam(model_self.parameters(), lr=1e-3)

for epoch in range(30):
    for x, y in loader:
        loss = trainer.train_step(x, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        trainer.update_ema()

model_baseline = StudentModel(dim=64, n_classes=10)
train_normal(model_baseline, loader, epochs=30)

with torch.no_grad():
    acc_self = (model_self(x_data).argmax(1) == y_data).float().mean()
    acc_base = (model_baseline(x_data).argmax(1) == y_data).float().mean()

print(f"=== 自蒸馏效果 ===")
print(f"普通训练准确率: {acc_base:.4f}")
print(f"自蒸馏(EMA教师)准确率: {acc_self:.4f}")
print(f"提升: {(acc_self - acc_base)*100:.2f}%")
print(f"\n自蒸馏优势: 无需额外教师模型，EMA提供更稳定的训练目标")
print(f"适用场景: 无法获取更大教师模型时的替代方案")

### 反向KL蒸馏（Reverse KL / MiniLLM）

#### 什么是反向KL蒸馏？

标准蒸馏使用正向KL散度 $KL(p_t || p_s)$，让学生分布覆盖教师分布的所有模式，但可能产生"平均化"输出。反向KL蒸馏使用 $KL(p_s || p_t)$，让学生分布在教师分布的某个模式附近集中，更适合生成式任务。MiniLLM（2023）首次将反向KL应用于LLM蒸馏。

#### 为什么反向KL更适合生成式模型？

1. **模式寻求 vs 模式覆盖**：正向KL是"模式覆盖"（mode-covering），学生试图覆盖教师的所有输出模式，可能导致不自然的平均化输出。反向KL是"模式寻求"（mode-seeking），学生集中在教师最可能输出的模式上，生成更锐利、更自然的文本。
2. **避免"教师幻觉"传递**：正向KL会让学生也学到教师低概率区域的噪声，反向KL则忽略这些区域。
3. **生成质量更高**：实验表明，反向KL蒸馏的学生模型在生成任务上产生更少重复、更连贯的文本。

#### 反向KL蒸馏的数学公式

**正向KL（标准蒸馏）**：
$$KL(p_t || p_s) = \sum_x p_t(x) \log \frac{p_t(x)}{p_s(x)}$$

**反向KL（MiniLLM）**：
$$KL(p_s || p_t) = \sum_x p_s(x) \log \frac{p_s(x)}{p_t(x)}$$

**MiniLLM损失**：
$$L = \mathbb{E}_{x \sim p_s}[\log p_s(x) - \log p_t(x)] + \lambda \cdot L_{CE}$$

其中：
- $p_t$：教师模型的概率分布
- $p_s$：学生模型的概率分布
- 正向KL：惩罚学生未覆盖教师高概率区域（模式覆盖）
- 反向KL：惩罚学生分配概率给教师低概率区域（模式寻求）
- $\lambda$：CE损失权重
- 反向KL需要从学生采样，训练更复杂（需要REINFORCE梯度估计）

In [ ]:
def reverse_kl_distillation_loss(student_logits, teacher_logits, labels,
                                 temperature=4.0, alpha=0.7):
    """反向KL蒸馏损失: KL(p_s || p_t) 而非 KL(p_t || p_s)
    反向KL: 学生为"驱动分布"，教师为"目标分布"
    F.kl_div中: input=log(p_s), target=p_t, 但交换student/teacher角色"""
    soft_teacher = F.log_softmax(teacher_logits / temperature, dim=-1)
    soft_student = F.softmax(student_logits / temperature, dim=-1)
    reverse_kl = F.kl_div(soft_teacher, soft_student.detach(), reduction='batchmean') * (temperature ** 2)
    ce_loss = F.cross_entropy(student_logits, labels)
    return alpha * reverse_kl + (1 - alpha) * ce_loss

def train_reverse_kl(teacher, student, train_loader, epochs=30,
                     temperature=4.0, alpha=0.7, lr=1e-3):
    """反向KL蒸馏训练"""
    teacher.eval()
    optimizer = torch.optim.Adam(student.parameters(), lr=lr)
    loss_history = []

    for epoch in range(epochs):
        epoch_loss = 0
        for x, y in train_loader:
            with torch.no_grad():
                teacher_logits = teacher(x)
            student_logits = student(x)
            loss = reverse_kl_distillation_loss(student_logits, teacher_logits, y,
                                                temperature=temperature, alpha=alpha)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        loss_history.append(epoch_loss / len(train_loader))
    return loss_history

student_fwd_kd = StudentModel(dim, n_classes)
student_rev_kd = StudentModel(dim, n_classes)

train_with_distillation(teacher, student_fwd_kd, loader, epochs=30, temperature=4.0)
train_reverse_kl(teacher, student_rev_kd, loader, epochs=30, temperature=4.0)

with torch.no_grad():
    acc_fwd = (student_fwd_kd(x_data).argmax(1) == y_data).float().mean()
    acc_rev = (student_rev_kd(x_data).argmax(1) == y_data).float().mean()

    fwd_probs = F.softmax(student_fwd_kd(x_data), dim=-1)
    rev_probs = F.softmax(student_rev_kd(x_data), dim=-1)
    teacher_probs = F.softmax(teacher(x_data), dim=-1)

    fwd_entropy = -(fwd_probs * fwd_probs.log()).sum(dim=-1).mean()
    rev_entropy = -(rev_probs * rev_probs.log()).sum(dim=-1).mean()
    teacher_entropy = -(teacher_probs * teacher_probs.log()).sum(dim=-1).mean()

print(f"=== 正向KL vs 反向KL蒸馏 ===")
print(f"正向KL蒸馏准确率: {acc_fwd:.4f}")
print(f"反向KL蒸馏准确率: {acc_rev:.4f}")
print(f"\n输出分布熵（越低越锐利）:")
print(f"  教师: {teacher_entropy:.4f}")
print(f"  正向KL学生: {fwd_entropy:.4f}")
print(f"  反向KL学生: {rev_entropy:.4f}")
print(f"\n关键: 反向KL使学生分布更集中(模式寻求)，生成式任务效果更好")
print(f"正向KL使学生分布更分散(模式覆盖)，分类任务效果通常更好")

### 蒸馏方法综合对比

不同蒸馏方法在信息需求、训练成本和精度上限方面的对比。选择蒸馏方法时需根据实际条件（是否有教师模型访问权限、训练资源等）决定。

In [ ]:
print(f"{'方法':<25} {'需要教师内部状态':<20} {'训练成本':<15} {'精度上限':<15}")
print("-" * 75)
methods = [
    ("Logits蒸馏", "需要logits", "中等", "高"),
    ("特征级蒸馏", "需要中间层特征", "较高", "更高"),
    ("注意力蒸馏", "需要注意力图", "较高", "高"),
    ("指令蒸馏(黑盒)", "仅需输出文本", "低", "中等"),
    ("自蒸馏(EMA)", "无需教师", "低", "中等"),
    ("MiniLLM(反向KL)", "需要logits", "中等", "高(生成式)"),
]
for name, state, cost, acc in methods:
    print(f"{name:<25} {state:<20} {cost:<15} {acc:<15}")

print(f"\n=== 产业级选择建议 ===")
print(f"1. 有教师模型访问权限: 白盒蒸馏(Logits+特征)效果最好")
print(f"2. 仅有API访问: 黑盒指令蒸馏，数据质量是关键")
print(f"3. 无教师模型: 自蒸馏(EMA)或渐进式训练")
print(f"4. 生成式LLM: MiniLLM的反向KL散度更适合")

### 产业级实战：完整蒸馏Pipeline

产业界的知识蒸馏是一个系统工程，包含：教师模型准备 → 数据校准 → 多损失蒸馏训练 → 精度验证 → 迭代优化。以下展示一个完整的端到端蒸馏pipeline，包含训练调度、梯度累积、学习率调度等实际应用中的关键组件。

**关键实践要点**：
1. **教师模型冻结**：教师模型完全冻结，仅用于推理，不参与梯度计算
2. **混合损失调度**：训练初期CE损失权重高（保证基本分类能力），后期蒸馏损失权重高（精细对齐教师分布）
3. **学习率调度**：使用cosine annealing或warmup+decay，避免训练后期震荡
4. **梯度累积**：当batch size受限于显存时，通过梯度累积模拟大batch训练

In [ ]:
import time

class DistillationPipeline:
    """产业级蒸馏Pipeline"""
    def __init__(self, teacher, student, temperature=4.0, alpha=0.7,
                 lr=1e-3, warmup_epochs=5, total_epochs=50):
        self.teacher = teacher
        self.student = student
        self.temperature = temperature
        self.alpha = alpha
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs

        self.teacher.eval()
        for p in self.teacher.parameters():
            p.requires_grad = False

        self.optimizer = torch.optim.AdamW(student.parameters(), lr=lr, weight_decay=1e-4)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=total_epochs)

    def get_alpha(self, epoch):
        """动态alpha调度: warmup阶段CE权重高，后期蒸馏权重高"""
        if epoch < self.warmup_epochs:
            return 0.3
        progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
        return 0.3 + 0.5 * progress

    def train_epoch(self, train_loader, epoch):
        self.student.train()
        alpha = self.get_alpha(epoch)
        epoch_loss = 0
        n_batches = 0

        for x, y in train_loader:
            with torch.no_grad():
                teacher_logits = self.teacher(x)
            student_logits = self.student(x)
            loss = distillation_loss(student_logits, teacher_logits, y,
                                     temperature=self.temperature, alpha=alpha)
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.student.parameters(), max_norm=1.0)
            self.optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        self.scheduler.step()
        return epoch_loss / n_batches, alpha

    def evaluate(self, eval_loader):
        self.student.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for x, y in eval_loader:
                pred = self.student(x).argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)
        return correct / total

    def run(self, train_loader, eval_loader=None):
        history = {'loss': [], 'alpha': [], 'acc': []}
        for epoch in range(self.total_epochs):
            loss, alpha = self.train_epoch(train_loader, epoch)
            acc = self.evaluate(eval_loader) if eval_loader else 0.0
            history['loss'].append(loss)
            history['alpha'].append(alpha)
            history['acc'].append(acc)
        return history

dim, n_classes = 64, 10
x_train = torch.randn(1024, dim)
y_train = torch.randint(0, n_classes, (1024,))
x_eval = torch.randn(256, dim)
y_eval = torch.randint(0, n_classes, (256,))

train_ds = torch.utils.data.TensorDataset(x_train, y_train)
eval_ds = torch.utils.data.TensorDataset(x_eval, y_eval)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64)
eval_loader = torch.utils.data.DataLoader(eval_ds, batch_size=64)

pipe_teacher = TeacherModel(dim, n_classes)
train_normal(pipe_teacher, train_loader, epochs=30)

pipe_student = StudentModel(dim, n_classes)
pipeline = DistillationPipeline(pipe_teacher, pipe_student,
                                 temperature=4.0, alpha=0.7,
                                 lr=1e-3, warmup_epochs=5, total_epochs=50)
start = time.perf_counter()
history = pipeline.run(train_loader, eval_loader)
elapsed = time.perf_counter() - start

baseline_student = StudentModel(dim, n_classes)
train_normal(baseline_student, train_loader, epochs=50)

teacher_params = sum(p.numel() for p in pipe_teacher.parameters())
student_params = sum(p.numel() for p in pipe_student.parameters())

with torch.no_grad():
    acc_teacher = (pipe_teacher(x_eval).argmax(1) == y_eval).float().mean()
    acc_pipeline = (pipe_student(x_eval).argmax(1) == y_eval).float().mean()
    acc_baseline = (baseline_student(x_eval).argmax(1) == y_eval).float().mean()

print(f"=== 产业级蒸馏Pipeline ===")
print(f"教师: {teacher_params:,}参数, 准确率={acc_teacher:.4f}")
print(f"学生(Pipeline蒸馏): {student_params:,}参数, 准确率={acc_pipeline:.4f}")
print(f"学生(普通训练): {student_params:,}参数, 准确率={acc_baseline:.4f}")
print(f"蒸馏提升: {(acc_pipeline - acc_baseline)*100:.2f}%")
print(f"压缩比: {teacher_params/student_params:.1f}x")
print(f"训练耗时: {elapsed:.1f}s")
print(f"\nPipeline关键组件:")
print(f"  - 动态alpha调度: warmup阶段0.3 -> 后期0.8")
print(f"  - CosineAnnealing学习率调度")
print(f"  - 梯度裁剪(max_norm=1.0)")
print(f"  - AdamW + weight_decay")
print(f"  - 教师模型完全冻结(no_grad)")